# 2 — Le tour de magie de Strassen (1969)

Deux matrices 2×2 demandent 8 multiplications scalaires avec la méthode scolaire.
Strassen en trouve 7. Appliqué récursivement, cela donne l'exposant log₂7 ≈ 2,807.

Puis on chronomètre : et on constate que Strassen est **plus lent** que NumPy.

In [ ]:
import sys
sys.path.insert(0, "../demo")
from strassen import (
    naive_products, strassen_products, strassen_additions,
    _strassen, _strassen_blas, _median_time,
)
import numpy as np, math

for k in range(1, 8):
    n = 2**k
    print(f"n={n:>4}  scolaire={naive_products(n):>12,}  strassen={strassen_products(n):>12,}")

In [ ]:
# Vérification exacte sur entiers
rng = np.random.default_rng(0)
A = rng.integers(-9, 9, (4, 4)).astype(np.int64)
B = rng.integers(-9, 9, (4, 4)).astype(np.int64)
C = _strassen(A, B, 1, [0])
print("égalité exacte :", np.array_equal(C, A @ B))

In [ ]:
# Le chrono qui contredit la théorie
for n in (256, 512, 1024):
    A = rng.standard_normal((n, n)); B = rng.standard_normal((n, n))
    t1 = _median_time(lambda: A @ B)
    t2 = _median_time(lambda: _strassen_blas(A, B, 256))
    print(f"n={n:>5}  BLAS {t1*1e3:7.1f} ms   Strassen {t2*1e3:7.1f} ms   x{t2/t1:.2f}")

## Pourquoi cet écart ?

| Cause | Effet |
|---|---|
| Copies de blocs | 7 allocations par niveau de récursion |
| Localité cache | Strassen saute entre 7 sous-blocs → cache froid |
| Vectorisation | BLAS fait 8 multiplications par cycle, Strassen non |
| Constante | `T(n) = C·n^2.807` avec C très grand |

Un exposant est une **pente asymptotique**. À n = 1000, on est encore dans la
partie dominée par la constante.